# Stacked Model: Sector-Specific CatBoost Models

## Architecture
```
Input row
  --> TR.GICSSectorCode --> sector router
                             - sector 10 model - |
                             - sector 15 model   |
                             - ...               |
                             - sector 60 model ----> final prediction
```

### Two stacking strategies are implemented here
1. **Hard routing** – each row goes exclusively to the matching sector model
   (fallback: global model for unseen sectors at inference time)
2. **Soft blending** – all sector models produce a prediction; a meta-learner
   (Ridge regression) combines them with the global model prediction

### Key design constraints carried over from your existing training
- GroupKFold by `Instrument` (no firm leaks across train/val)
- Log-transformed target `TR.UpstreamScope3PurchasedGoodsAndServices`
- Categorical features: `TR.GICSSectorCode`, `TR.HQCountryCode`
- `Instrument` used only for grouping, dropped from features
- Depth reduced (see `SECTOR_DEPTH`) – this is what made sector models competitive


In [1]:
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from catboost import Pool, CatBoostRegressor

from core import Config

config = Config()

# ── Dataset ────────────────────────────────────────────────────────────────
DATASET: str = 'imputed_drop_zero_value_scope31'
DATA_PATH: Path = config.training_dir / f'{DATASET}.csv'
RESULTS_NAME: str = 'stacked_sector_catboost'

# ── Hyperparameters ────────────────────────────────────────────────────────
FOLDS: int = 5

# Global fallback model (same as your current best)
GLOBAL_DEPTH: int = 14
GLOBAL_ITERATIONS: int = 300
GLOBAL_LR: float = 0.05

# Sector-specific models (lower depth = less overfitting on small subsets)
SECTOR_DEPTH: int = 6  # ← tune this; your experiments showed ~6-8 works well
SECTOR_ITERATIONS: int = 300  # more iterations compensate for shallower trees
SECTOR_LR: float = 0.03
EARLY_STOPPING: int = 50

# Minimum samples in a sector to train a dedicated model;
# sectors below this threshold fall back to the global model.
MIN_SECTOR_SAMPLES: int = 30

TARGET: str = 'TR.UpstreamScope3PurchasedGoodsAndServices'
CAT_COLS: list[str] = ['TR.GICSSectorCode', 'TR.HQCountryCode']

In [2]:
# ── Load & prepare ─────────────────────────────────────────────────────────
all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)

training_data = all_data.dropna(subset=[TARGET]).copy()
training_data = training_data[training_data[TARGET] != 0]

y: pd.Series = training_data[TARGET]
X: pd.DataFrame = training_data.drop(columns=[TARGET])
X[CAT_COLS] = X[CAT_COLS].astype(str)

SECTORS: list[str] = sorted(X['TR.GICSSectorCode'].unique().tolist())
print(f"Loaded {len(X)} rows | {len(SECTORS)} sectors: {SECTORS}")

Loaded 11104 rows | 11 sectors: ['10', '15', '20', '25', '30', '35', '40', '45', '50', '55', '60']


---
## Strategy 1 – Hard Routing

Each row is predicted by the model that was trained exclusively on its sector.  
Rows from sectors with too few samples use the global model as fallback.

**Cross-validation design:**  
For each outer fold:
1. Train one *global* model on all training rows (fallback).
2. For each sector present in the training split:  
   a. Filter training rows to that sector.  
   b. Train a sector-specific model.  
3. Route each validation row to its sector model (or global fallback).
4. Collect predictions and compute metrics.

> **⚠️ Leakage note:** The val_pool for early stopping of sector models  
> is built from the sector slice of the *outer* validation fold.  
> This is fine because no sector-model ever sees full validation labels.  
> The global model uses a small held-out slice from its own training data  
> for early stopping (see `global_es_fraction`).

In [3]:
def make_pool(X_part: pd.DataFrame, y_part: pd.Series) -> Pool:
    return Pool(data=X_part, label=y_part, cat_features=CAT_COLS)


def train_catboost(
        train_pool: Pool,
        val_pool: Pool,
        depth: int,
        iterations: int,
        lr: float,
) -> CatBoostRegressor:
    model = CatBoostRegressor(
        iterations=iterations,
        learning_rate=lr,
        depth=depth,
        loss_function='RMSE',
        eval_metric='R2',
        use_best_model=True,
        verbose=False,
        early_stopping_rounds=EARLY_STOPPING,
    )
    model.fit(train_pool, eval_set=val_pool)
    return model


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }


# ── Outer CV loop ──────────────────────────────────────────────────────────
gkf = GroupKFold(n_splits=FOLDS)
X_no_inst = X.drop(columns=['Instrument'])

fold_results_hard: list[dict] = []

# Keep the best sector-model set (by overall RMSE) for later inspection / SHAP
best_hard_rmse: float | None = None
best_hard_sector_models: dict[str, CatBoostRegressor] = {}
best_hard_global_model: CatBoostRegressor | None = None

for fold_idx, (train_idx, val_idx) in enumerate(
        gkf.split(X_no_inst, y, groups=X['Instrument']), start=1
):
    print(f"\n── Fold {fold_idx} ──")

    X_tr = cast(pd.DataFrame, X_no_inst.iloc[train_idx])
    X_va = cast(pd.DataFrame, X_no_inst.iloc[val_idx])
    y_tr = cast(pd.Series, y.iloc[train_idx])
    y_va = cast(pd.Series, y.iloc[val_idx])

    # ── 1. Global fallback model ────────────────────────────────────────────
    # Use 10 % of training data as an internal early-stopping set for global model
    global_es_fraction = 0.1
    n_es = max(1, int(len(X_tr) * global_es_fraction))
    X_tr_main, X_tr_es = X_tr.iloc[:-n_es], X_tr.iloc[-n_es:]
    y_tr_main, y_tr_es = y_tr.iloc[:-n_es], y_tr.iloc[-n_es:]

    global_model = train_catboost(
        make_pool(X_tr_main, y_tr_main),
        make_pool(X_tr_es, y_tr_es),
        depth=GLOBAL_DEPTH,
        iterations=GLOBAL_ITERATIONS,
        lr=GLOBAL_LR,
    )
    print("  Global model trained.")

    # ── 2. Sector-specific models ───────────────────────────────────────────
    sector_models: dict[str, CatBoostRegressor] = {}

    for sector in SECTORS:
        tr_mask = X_tr['TR.GICSSectorCode'].astype(str) == str(sector)
        va_mask = X_va['TR.GICSSectorCode'].astype(str) == str(sector)

        X_tr_s = X_tr[tr_mask]
        y_tr_s = y_tr[tr_mask]
        X_va_s = X_va[va_mask]
        y_va_s = y_va[va_mask]

        if len(X_tr_s) < MIN_SECTOR_SAMPLES or len(X_va_s) == 0:
            print(f"  Sector {sector}: too few samples ({len(X_tr_s)} train, "
                  f"{len(X_va_s)} val) → using global fallback")
            continue

        sector_model = train_catboost(
            make_pool(X_tr_s, y_tr_s),
            make_pool(X_va_s, y_va_s),
            depth=SECTOR_DEPTH,
            iterations=SECTOR_ITERATIONS,
            lr=SECTOR_LR,
        )
        sector_models[str(sector)] = sector_model
        print(f"  Sector {sector}: trained (best iter = "
              f"{sector_model.get_best_iteration()})")

    # ── 3. Hard-route validation predictions ───────────────────────────────
    y_pred_hard = np.full(len(y_va), np.nan)

    for sector in SECTORS:
        va_mask = X_va['TR.GICSSectorCode'].astype(str) == str(sector)
        if va_mask.sum() == 0:
            continue

        X_va_s = X_va[va_mask]
        model_to_use = sector_models.get(str(sector), global_model)
        pool_s = make_pool(X_va_s, y_va[va_mask])
        preds_s = model_to_use.predict(pool_s)
        y_pred_hard[np.where(va_mask.values)[0]] = preds_s

    # Fill any remaining NaN with global model (should not happen normally)
    remaining = np.isnan(y_pred_hard)
    if remaining.any():
        global_pool_rem = make_pool(X_va[remaining], y_va.values[remaining])
        y_pred_hard[remaining] = global_model.predict(global_pool_rem)

    y_va_arr = y_va.to_numpy().ravel()

    # ── 4. Metrics ─────────────────────────────────────────────────────────
    m_all = compute_metrics(y_va_arr, y_pred_hard)
    fold_results_hard.append({'model_name': RESULTS_NAME + '_hard',
                              'fold': fold_idx, 'sector': 'All', **m_all})
    print(f"  All  → RMSE={m_all['rmse']:.4f}  MAE={m_all['mae']:.4f}  "
          f"R²={m_all['r2']:.4f}")

    val_df = pd.DataFrame({'y_true': y_va_arr, 'y_pred': y_pred_hard,
                           'sector': X_va['TR.GICSSectorCode'].values})
    for sector, grp in val_df.groupby('sector'):
        m_s = compute_metrics(grp['y_true'].values, grp['y_pred'].values)
        fold_results_hard.append({'model_name': RESULTS_NAME + '_hard',
                                  'fold': fold_idx, 'sector': sector, **m_s})

    # ── 5. Track best fold ─────────────────────────────────────────────────
    if best_hard_rmse is None or m_all['rmse'] < best_hard_rmse:
        best_hard_rmse = m_all['rmse']
        best_hard_sector_models = sector_models
        best_hard_global_model = global_model

print(f"\nBest hard-routing RMSE: {best_hard_rmse:.4f}")
metrics_hard = pd.DataFrame(fold_results_hard)


── Fold 1 ──
  Global model trained.
  Sector 10: trained (best iter = 81)
  Sector 15: trained (best iter = 160)
  Sector 20: trained (best iter = 293)
  Sector 25: trained (best iter = 127)
  Sector 30: trained (best iter = 67)
  Sector 35: trained (best iter = 279)
  Sector 40: trained (best iter = 163)
  Sector 45: trained (best iter = 192)
  Sector 50: trained (best iter = 178)
  Sector 55: trained (best iter = 56)
  Sector 60: trained (best iter = 135)
  All  → RMSE=1.8817  MAE=1.3326  R²=0.6691

── Fold 2 ──
  Global model trained.
  Sector 10: trained (best iter = 155)
  Sector 15: trained (best iter = 280)
  Sector 20: trained (best iter = 267)
  Sector 25: trained (best iter = 293)
  Sector 30: trained (best iter = 178)
  Sector 35: trained (best iter = 224)
  Sector 40: trained (best iter = 131)
  Sector 45: trained (best iter = 213)
  Sector 50: trained (best iter = 247)
  Sector 55: trained (best iter = 69)
  Sector 60: trained (best iter = 211)
  All  → RMSE=1.7686  MAE=

---
## Strategy 2 – Soft Blending (Meta-Learner)

All sector models produce a prediction for every row.  
A Ridge regression meta-learner learns the optimal linear combination.

**Cross-validation design (nested):**
```
Outer fold
  ├─ Training split  ──► inner CV  ──► out-of-fold predictions for meta-features
  │                                    (avoids leakage into meta-learner)
  └─ Validation split ──► retrain all base models on full training split
                          ──► generate meta-features for validation rows
                          ──► meta-learner predicts final output
```

This is standard **stacked generalization** (Wolpert 1992).  
The inner loop generates unbiased base-model predictions on the training set
that the meta-learner trains on.

In [4]:
import pickle
from pathlib import Path


# ── Wrapper class ──────────────────────────────────────────────────────────
class SoftBlendedSectorModel:
    def __init__(self, sector_models, global_model, meta_model, meta_cols):
        self.sector_models = sector_models  # dict[str, CatBoostRegressor]
        self.global_model = global_model  # CatBoostRegressor
        self.meta_model = meta_model  # fitted Ridge
        self.meta_cols = meta_cols  # list[str] — column order Ridge expects

    def _build_meta_features(self, X, y):
        val_meta = pd.DataFrame(np.nan, index=range(len(X)), columns=self.meta_cols)
        val_meta['global'] = self.global_model.predict(make_pool(X, y))
        for sector in SECTORS:
            va_mask = X['TR.GICSSectorCode'].astype(str) == str(sector)
            model_to_use = self.sector_models.get(str(sector), self.global_model)
            if va_mask.sum() == 0:
                val_meta[f's_{sector}'] = val_meta['global']
                continue
            preds = model_to_use.predict(make_pool(X[va_mask], y[va_mask]))
            val_meta.loc[va_mask.values, f's_{sector}'] = preds
            val_meta.loc[~va_mask.values, f's_{sector}'] = val_meta.loc[
                ~va_mask.values, 'global'
            ].values
        return val_meta

    def predict(self, X, y=None):
        if y is None:
            y = pd.Series(np.zeros(len(X)), index=X.index)
        val_meta = self._build_meta_features(X, y)
        return self.meta_model.predict(val_meta.values)

    def save(self, path):
        with open(Path(path), 'wb') as f:
            pickle.dump(self, f)
        print(f"  Saved soft-blended model → {path}")

    @staticmethod
    def load(path):
        with open(Path(path), 'rb') as f:
            return pickle.load(f)


# ── Inner fold count for meta-feature generation ───────────────────────────
INNER_FOLDS: int = 3

fold_results_soft: list[dict] = []
best_soft_rmse: float | None = None
best_soft_model: SoftBlendedSectorModel | None = None

for fold_idx, (train_idx, val_idx) in enumerate(
        gkf.split(X_no_inst, y, groups=X['Instrument']), start=1
):
    print(f"\n══ Outer fold {fold_idx} ══")

    X_tr = cast(pd.DataFrame, X_no_inst.iloc[train_idx])
    X_va = cast(pd.DataFrame, X_no_inst.iloc[val_idx])
    y_tr = cast(pd.Series, y.iloc[train_idx])
    y_va = cast(pd.Series, y.iloc[val_idx])
    groups_tr = X.iloc[train_idx]['Instrument']

    # ── Step A: Generate OOF meta-features on training split ───────────────
    meta_cols = ['global'] + [f's_{s}' for s in SECTORS]
    oof_meta = pd.DataFrame(np.nan, index=range(len(X_tr)), columns=meta_cols)
    inner_gkf = GroupKFold(n_splits=INNER_FOLDS)

    for inner_idx, (inn_tr_idx, inn_va_idx) in enumerate(
            inner_gkf.split(X_tr, y_tr, groups=groups_tr), start=1
    ):
        print(f"  Inner fold {inner_idx}")
        X_inn_tr = X_tr.iloc[inn_tr_idx]
        X_inn_va = X_tr.iloc[inn_va_idx]
        y_inn_tr = y_tr.iloc[inn_tr_idx]
        y_inn_va = y_tr.iloc[inn_va_idx]

        n_es = max(1, int(len(X_inn_tr) * 0.1))
        g_model = train_catboost(
            make_pool(X_inn_tr.iloc[:-n_es], y_inn_tr.iloc[:-n_es]),
            make_pool(X_inn_tr.iloc[-n_es:], y_inn_tr.iloc[-n_es:]),
            depth=GLOBAL_DEPTH, iterations=GLOBAL_ITERATIONS, lr=GLOBAL_LR,
        )
        oof_meta.loc[inn_va_idx, 'global'] = g_model.predict(
            make_pool(X_inn_va, y_inn_va)
        )

        for sector in SECTORS:
            tr_mask = X_inn_tr['TR.GICSSectorCode'].astype(str) == str(sector)
            va_mask = X_inn_va['TR.GICSSectorCode'].astype(str) == str(sector)

            if tr_mask.sum() < MIN_SECTOR_SAMPLES or va_mask.sum() == 0:
                oof_meta.loc[
                    [inn_va_idx[i] for i, m in enumerate(va_mask.values) if m],
                    f's_{sector}'
                ] = oof_meta.loc[
                    [inn_va_idx[i] for i, m in enumerate(va_mask.values) if m],
                    'global'
                ].values
                continue

            s_model = train_catboost(
                make_pool(X_inn_tr[tr_mask], y_inn_tr[tr_mask]),
                make_pool(X_inn_va[va_mask], y_inn_va[va_mask]),
                depth=SECTOR_DEPTH, iterations=SECTOR_ITERATIONS, lr=SECTOR_LR,
            )
            va_positions = [inn_va_idx[i] for i, m in enumerate(va_mask.values) if m]
            oof_meta.loc[va_positions, f's_{sector}'] = s_model.predict(
                make_pool(X_inn_va[va_mask], y_inn_va[va_mask])
            )

    oof_meta = oof_meta.fillna(oof_meta.mean())

    # ── Step B: Train meta-learner on OOF predictions ──────────────────────
    meta_learner = Ridge(alpha=1.0)
    meta_learner.fit(oof_meta.values, y_tr.values)
    print(f"  Meta-learner coefficients: "
          f"{dict(zip(meta_cols, meta_learner.coef_.round(3)))}")

    # ── Step C: Retrain ALL base models on full training split ──────────────
    n_es = max(1, int(len(X_tr) * 0.1))
    final_global = train_catboost(
        make_pool(X_tr.iloc[:-n_es], y_tr.iloc[:-n_es]),
        make_pool(X_tr.iloc[-n_es:], y_tr.iloc[-n_es:]),
        depth=GLOBAL_DEPTH, iterations=GLOBAL_ITERATIONS, lr=GLOBAL_LR,
    )

    final_sector_models: dict[str, CatBoostRegressor] = {}
    for sector in SECTORS:
        tr_mask = X_tr['TR.GICSSectorCode'].astype(str) == str(sector)
        va_mask = X_va['TR.GICSSectorCode'].astype(str) == str(sector)
        if tr_mask.sum() < MIN_SECTOR_SAMPLES or va_mask.sum() == 0:
            continue
        final_sector_models[str(sector)] = train_catboost(
            make_pool(X_tr[tr_mask], y_tr[tr_mask]),
            make_pool(X_va[va_mask], y_va[va_mask]),
            depth=SECTOR_DEPTH, iterations=SECTOR_ITERATIONS, lr=SECTOR_LR,
        )

    # ── Step D: Build meta-features for validation set ─────────────────────
    val_meta = pd.DataFrame(np.nan, index=range(len(X_va)), columns=meta_cols)
    val_meta['global'] = final_global.predict(make_pool(X_va, y_va))

    for sector in SECTORS:
        va_mask = X_va['TR.GICSSectorCode'].astype(str) == str(sector)
        model_to_use = final_sector_models.get(str(sector), final_global)
        if va_mask.sum() == 0:
            val_meta[f's_{sector}'] = val_meta['global']
            continue
        preds = model_to_use.predict(make_pool(X_va[va_mask], y_va[va_mask]))
        val_meta.loc[va_mask.values, f's_{sector}'] = preds
        val_meta.loc[~va_mask.values, f's_{sector}'] = val_meta.loc[
            ~va_mask.values, 'global'
        ].values

    # ── Step E: Final prediction via meta-learner ───────────────────────────
    y_pred_soft = meta_learner.predict(val_meta.values)
    y_va_arr = y_va.to_numpy().ravel()

    m_all = compute_metrics(y_va_arr, y_pred_soft)
    fold_results_soft.append({'model_name': RESULTS_NAME + '_soft',
                              'fold': fold_idx, 'sector': 'All', **m_all})
    print(f"  All  → RMSE={m_all['rmse']:.4f}  MAE={m_all['mae']:.4f}  "
          f"R²={m_all['r2']:.4f}")

    val_df = pd.DataFrame({'y_true': y_va_arr, 'y_pred': y_pred_soft,
                           'sector': X_va['TR.GICSSectorCode'].values})
    for sector, grp in val_df.groupby('sector'):
        m_s = compute_metrics(grp['y_true'].values, grp['y_pred'].values)
        fold_results_soft.append({'model_name': RESULTS_NAME + '_soft',
                                  'fold': fold_idx, 'sector': sector, **m_s})

    # ── Step F: Track best fold and save ───────────────────────────────────
    if best_soft_rmse is None or m_all['rmse'] < best_soft_rmse:
        best_soft_rmse = m_all['rmse']
        best_soft_model = SoftBlendedSectorModel(
            sector_models=final_sector_models,
            global_model=final_global,
            meta_model=meta_learner,
            meta_cols=meta_cols,
        )
        best_soft_model.save(
            config.results_dir / f'model_{RESULTS_NAME}_soft_best.pkl'
        )
        print(f"  ★ New best soft model (fold {fold_idx}, RMSE={best_soft_rmse:.4f})")

metrics_soft = pd.DataFrame(fold_results_soft)


══ Outer fold 1 ══
  Inner fold 1
  Inner fold 2
  Inner fold 3
  Meta-learner coefficients: {'global': np.float64(0.976), 's_10': np.float64(0.327), 's_15': np.float64(0.149), 's_20': np.float64(0.179), 's_25': np.float64(0.163), 's_30': np.float64(0.011), 's_35': np.float64(0.205), 's_40': np.float64(0.22), 's_45': np.float64(0.081), 's_50': np.float64(0.128), 's_55': np.float64(0.203), 's_60': np.float64(0.186)}
  All  → RMSE=4.6772  MAE=3.8127  R²=-1.0446
  Saved soft-blended model → /Users/sebastianbecker/PycharmProjects/ML/data/results/model_stacked_sector_catboost_soft_best.pkl
  ★ New best soft model (fold 1, RMSE=4.6772)

══ Outer fold 2 ══
  Inner fold 1
  Inner fold 2
  Inner fold 3
  Meta-learner coefficients: {'global': np.float64(0.997), 's_10': np.float64(0.283), 's_15': np.float64(0.061), 's_20': np.float64(0.096), 's_25': np.float64(0.082), 's_30': np.float64(0.065), 's_35': np.float64(0.175), 's_40': np.float64(0.145), 's_45': np.float64(0.009), 's_50': np.float64(0.

In [5]:
# ── Aggregate: mean ± std across folds ─────────────────────────────────────
def summarise(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby('sector')[['rmse', 'mae', 'r2']]
        .agg(['mean', 'std'])
        .round(4)
    )


summary_hard = summarise(metrics_hard)
summary_soft = summarise(metrics_soft)

print("=== Hard-Routing Summary ===")
print(summary_hard)

print("\n=== Soft-Blending Summary ===")
print(summary_soft)

=== Hard-Routing Summary ===
          rmse             mae              r2        
          mean     std    mean     std    mean     std
sector                                                
10      1.9797  0.4746  1.4577  0.2312  0.5552  0.0777
15      1.3366  0.2654  0.8947  0.1395  0.5036  0.0663
20      1.7004  0.0846  1.1614  0.0468  0.5242  0.0558
25      1.7087  0.1481  1.1177  0.0950  0.5643  0.0664
30      1.8883  0.3053  1.1586  0.1570  0.4415  0.1545
35      1.5104  0.4576  1.0495  0.2495  0.6866  0.0938
40      2.2078  0.1750  1.6886  0.1417  0.4569  0.0191
45      1.9041  0.0910  1.3560  0.0395  0.5750  0.0494
50      1.9682  0.4735  1.4319  0.3221  0.4558  0.2047
55      2.3826  0.2786  1.6975  0.3032  0.3567  0.0816
60      2.4414  0.1647  1.8726  0.1576  0.3450  0.1457
All     1.8874  0.0941  1.2990  0.0516  0.6672  0.0214

=== Soft-Blending Summary ===
          rmse             mae              r2        
          mean     std    mean     std    mean     std
secto

In [6]:
# ── Save results ────────────────────────────────────────────────────────────
metrics_hard.to_csv(config.results_dir / f'{RESULTS_NAME}_hard_metrics.csv')
metrics_soft.to_csv(config.results_dir / f'{RESULTS_NAME}_soft_metrics.csv')

# Save best hard-routing sector models
model_dir: Path = config.results_dir / 'model'
model_dir.mkdir(exist_ok=True)

for sector, model in best_hard_sector_models.items():
    model.save_model(str(model_dir / f'{RESULTS_NAME}_sector_{sector}.bin'))

if best_hard_global_model is not None:
    best_hard_global_model.save_model(
        str(model_dir / f'{RESULTS_NAME}_global_fallback.bin')
    )

print("All models and metrics saved.")

All models and metrics saved.


In [7]:
# ── Inference helper (hard-routing) ─────────────────────────────────────────
# After training, load saved models and use this class for new data.

class StackedSectorPredictor:
    """Hard-routing stacked predictor. Routes each row to its sector model."""

    def __init__(
            self,
            sector_models: dict[str, CatBoostRegressor],
            global_model: CatBoostRegressor,
            cat_features: list[str],
    ):
        self.sector_models = sector_models
        self.global_model = global_model
        self.cat_features = cat_features

    @classmethod
    def load(
            cls,
            model_dir: Path,
            results_name: str,
            sectors: list[str],
            cat_features: list[str],
    ) -> 'StackedSectorPredictor':
        global_model = CatBoostRegressor()
        global_model.load_model(
            str(model_dir / f'{results_name}_global_fallback.bin')
        )
        sector_models: dict[str, CatBoostRegressor] = {}
        for s in sectors:
            p = model_dir / f'{results_name}_sector_{s}.bin'
            if p.exists():
                m = CatBoostRegressor()
                m.load_model(str(p))
                sector_models[str(s)] = m
        return cls(sector_models, global_model, cat_features)

    def predict(self, X_new: pd.DataFrame) -> np.ndarray:
        """Predict on a new DataFrame (must contain TR.GICSSectorCode)."""
        X_new = X_new.copy()
        X_new[self.cat_features] = X_new[self.cat_features].astype(str)
        preds = np.full(len(X_new), np.nan)

        for sector in X_new['TR.GICSSectorCode'].unique():
            mask = X_new['TR.GICSSectorCode'] == str(sector)
            model = self.sector_models.get(str(sector), self.global_model)
            pool = Pool(data=X_new[mask], cat_features=self.cat_features)
            preds[mask.values] = model.predict(pool)

        return preds

# Example usage after training:
# predictor = StackedSectorPredictor.load(
#     model_dir=config.results_dir / 'model',
#     results_name=RESULTS_NAME,
#     sectors=SECTORS,
#     cat_features=CAT_COLS,
# )
# y_hat = predictor.predict(X_new_data)

In [13]:
val = pd.read_csv(config.dataset_dir / 'verification.csv', index_col=0)
val.dropna(subset=[TARGET], inplace=True)
y_val = val[TARGET].to_numpy().ravel()
X_val = val.drop(columns=['Instrument', 'TR.NumberofEmployees', TARGET])
val_pool = Pool(data=X_val, label=y_val, cat_features=CAT_COLS)

hard_predictor = StackedSectorPredictor.load(
    model_dir=config.results_dir / 'model',
    results_name=RESULTS_NAME,
    sectors=SECTORS,
    cat_features=CAT_COLS,
)
y_pred = hard_predictor.predict(X_val)
rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
mae = float(mean_absolute_error(y_val, y_pred))
r2 = float(r2_score(y_val, y_pred))

print(f"Validation Hard Routing RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

Validation Hard Routing RMSE=1.293, MAE=0.901, R2=0.793


In [14]:
best_predictor = CatBoostRegressor().load_model(
    f"{config.results_dir}/model/metrics_catboost_14_imputed_drop_zero_value_scope31_200_iter_model.bin"
)
y_pred = best_predictor.predict(X_val)
rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
mae = float(mean_absolute_error(y_val, y_pred))
r2 = float(r2_score(y_val, y_pred))

print(f"Validation Best Global RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

Validation Best Global RMSE=1.180, MAE=0.783, R2=0.828


In [15]:
soft_predictor = SoftBlendedSectorModel.load(
    config.results_dir / f'model_{RESULTS_NAME}_soft_best.pkl'
)
y_pred = soft_predictor.predict(X_val)
rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
mae = float(mean_absolute_error(y_val, y_pred))
r2 = float(r2_score(y_val, y_pred))

print(f"Validation Soft-Blending RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}")

Validation Soft-Blending RMSE=3.194, MAE=2.566, R2=-0.261
